# Extract residual-stream activations — Qwen2.5-1.5B (Colab)

Runs on Colab GPU. Extracts `resid_post` activations at the **last token**
for every statement in the Geometry of Truth datasets and saves them to
Google Drive as `.npy` files, to be pulled down and probed locally on CPU.

Model loading and extraction now live in `src/extract.py`, and dataset
loading in `src/data.py` (this repo is cloned below alongside Geometry of
Truth) -- this notebook just calls into them.

Conventions (see project `CLAUDE.md`):
- Activations shape `[n_statements, n_layers, d_model]`.
- Model is the **base** (non-instruct) Qwen2.5-1.5B, loaded via HF handoff
  into TransformerLens.
- Sign convention (positive = true/honest) is applied later, at probing time.

## 1. Mount Drive and set the save path

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# All activations/labels get written here, then pulled down to
# data/activations/ locally for the (CPU-only) probing side of the project.
SAVE_DIR = "/content/drive/MyDrive/bluedot_project/"
import os; os.makedirs(SAVE_DIR, exist_ok=True)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Install dependencies

In [ ]:
# transformer_lens gives cache access to resid_post at every layer in one
# forward pass; scikit-learn/pandas/matplotlib are for the local probing side.
!pip install transformer_lens scikit-learn pandas matplotlib


## 3. Clone this repo and the Geometry of Truth datasets

In [ ]:
!git clone https://github.com/graceshan/truth-probing-extensions.git
!git clone https://github.com/saprmarks/geometry-of-truth.git

import sys
sys.path.append("truth-probing-extensions")


## 4. Imports

In [ ]:
import torch
import numpy as np

from src.data import load_statements
from src.extract import load_model, extract_acts

device = "cuda" if torch.cuda.is_available() else "cpu"


## 5. Sanity-check a dataset

In [ ]:
# Sanity check: columns are `statement` and `label` (1 = true), per CLAUDE.md.
cities = load_statements("cities", datasets_dir="geometry-of-truth/datasets")
print(cities[["statement", "label"]].head(10)); print(len(cities))


fatal: destination path 'geometry-of-truth' already exists and is not an empty directory.
                                        statement  label
0             The city of Krasnodar is in Russia.      1
1       The city of Krasnodar is in South Africa.      0
2                  The city of Lodz is in Poland.      1
3  The city of Lodz is in the Dominican Republic.      0
4            The city of Maracay is in Venezuela.      1
5                The city of Maracay is in China.      0
6              The city of Baku is in Azerbaijan.      1
7                 The city of Baku is in Ukraine.      0
8                  The city of Baoji is in China.      1
9              The city of Baoji is in Guatemala.      0
1496


## 6. Load the model — base Qwen2.5-1.5B via HF handoff

In [ ]:
model = load_model(device=device)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loaded pretrained model qwen2.5-1.5b into HookedTransformer


## 7. Extract and save activations for each dataset

In [ ]:
# cities / neg_cities / sp_en_trans: matched true/false statement pairs used
# for the contrast-pair direction comparison.
for name in ["cities", "neg_cities", "sp_en_trans"]:
    df = load_statements(name, datasets_dir="geometry-of-truth/datasets")
    acts = extract_acts(model, df["statement"].tolist())
    np.save(SAVE_DIR + f"{name}_acts.npy", acts)
    np.save(SAVE_DIR + f"{name}_labels.npy", df["label"].to_numpy())


## 8. Extract and save activations for the compound and/or dataset

`compound_cities.csv` (generated locally by `scripts/00_generate_compounds.py`,
not part of the geometry-of-truth repo) lives in this repo's `data/`
instead, so it needs its own `load_statements` call with a different
`datasets_dir`. Saved with matching names (`compound_cities_acts.npy` /
`compound_cities_labels.npy`) so `load_activations("compound_cities")`
works the same way it does for the other three datasets.

In [ ]:
name = "compound_cities"
df = load_statements(name, datasets_dir="truth-probing/data")
acts = extract_acts(model, df["statement"].tolist())
np.save(SAVE_DIR + f"{name}_acts.npy", acts)
np.save(SAVE_DIR + f"{name}_labels.npy", df["label"].to_numpy())


## 9. Fact-check: does the model know these cities' countries?

Compositional truth (and/or) is only representable if the model knows
the underlying single-statement facts. Rather than checking the format
of a generated completion (which conflates instruction-following with
factual belief), compare log-likelihood directly: for each city, does
the model assign higher probability to the true statement than to every
false one? Runs over every city in the dataset, not just a sample --
cheap at this model size, and gives a reusable known/unknown label per
city rather than a one-off spot check.

In [ ]:
import numpy as np

from src.extract import statement_logprob

cities_df = load_statements("cities", datasets_dir="geometry-of-truth/datasets")
cities_df["city"] = cities_df["statement"].str.extract(r"city of (.+?) is in")

known = {}
for city, grp in cities_df.groupby("city"):
    true_stmt   = grp[grp.label == 1]["statement"].iloc[0]
    false_stmts = grp[grp.label == 0]["statement"].tolist()
    lp_true = statement_logprob(model, true_stmt)
    known[city] = all(lp_true > statement_logprob(model, f) for f in false_stmts)

print(f"knows {sum(known.values())}/{len(known)} = {np.mean(list(known.values())):.1%}")


## 10. Scrambled control: random-wrong-country baseline

Section 9's false statements are the dataset's own near-miss
alternatives (plausible wrong countries, sometimes same region). As a
control, compare against a statement with a *guaranteed*-wrong,
randomly chosen country instead -- this should be easy mode relative
to section 9, so a much lower win rate here would flag a bug rather
than a fact the model genuinely doesn't know.

In [ ]:
cities_df["country"] = cities_df["statement"].str.extract(r"is in (.+?)\.")

rng = np.random.default_rng(0)
true_rows = cities_df[cities_df.label == 1].copy()
all_countries = true_rows["country"].unique()

wins, tot = 0, 0
for _, row in true_rows.iterrows():
    # pick a country that isn't the true one
    choices = all_countries[all_countries != row["country"]]
    fake = rng.choice(choices)

    fake_stmt = f"The city of {row['city']} is in {fake}."
    wins += statement_logprob(model, row["statement"]) > statement_logprob(model, fake_stmt)
    tot += 1

print(f"scrambled control (guaranteed-wrong countries): {wins/tot:.1%}  ({wins}/{tot})")